<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_81_capstone_enterprise_ai_platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🏁 **Module 81 — Capstone: Enterprise AI Platform** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 10 · Master-Plan Gaps · **Capstone**


# Module 81 — Capstone: Enterprise AI Platform

> 80 modules of fragments. This one is **the platform** — a concrete blueprint that **wires every prior module into one running system**. The premise: you've been hired as platform lead at **Acme Insights**, a 200-employee B2B SaaS, to ship the AI Assistant their enterprise customers (hospitals, banks, retailers) are demanding. You have **one engineer per layer**, **$2M Year-1 cloud budget**, **6 months to GA**.
>
> This module isn't another concept primer — it's the **decision log + reference architecture + 26-week build plan** that says *use this module here*. By the end you can scope, staff, and ship this in your own org. It's the portfolio piece a hiring manager actually wants to see.

### What you'll cover
1. The brief — Acme's product, customers, constraints
2. The reference architecture (one picture you'll keep next to your desk)
3. Layer-by-layer module mapping (which of M1-M80 each layer pulls from)
4. The repo layout — what lives where
5. A live request walkthrough — 17 hops with timing budget
6. The 26-week build plan — quarter-by-quarter milestones
7. The team you actually need
8. Day-1 launch checklist (security + ops gates)
9. Day-90 evolution — what to add next
10. Failure modes that will hit you + the lessons we can pre-load


## 1 · The brief

**Acme Insights**, a 200-employee B2B SaaS, sells a workflow tool to hospitals, regional banks, and large retailers. Customers are asking for an **AI assistant** that:

- **Answers questions** about their own internal docs (RAG) — every customer has a unique corpus.
- **Drafts** emails, reports, summaries — domain-tuned (per customer LoRA).
- **Takes actions** (creates tickets, schedules tasks, calls APIs) — agentic, with HITL approval.
- **Ingests data** from their existing systems (Salesforce, ServiceNow, Slack, Drive).
- **Audits everything** — banks + hospitals need SOC2 / HIPAA / FedRAMP-light compliance.
- **Costs <$2/active user/month at scale.**

| Constraint | Value |
|---|---|
| **Customers Year 1** | 12 paid + 50 trial |
| **DAUs Year 1 target** | ~50K |
| **Budget Year 1** | $2.0M cloud + $1.6M payroll |
| **Time to GA** | 6 months |
| **Team size** | 8 engineers (1 lead + 7 specialists) |
| **Cloud** | AWS primary; Azure for one regulated customer |
| **Regulatory** | SOC2 Type 2, HIPAA-eligible, GDPR-compliant |
| **SLA** | p99 chat latency < 2.5s; 99.9% uptime |

These constraints **drive every architecture decision** below. Most of M1-M80 maps directly onto one of them.

## 2 · The reference architecture

```
                                    USERS  ◄──── M54 TypeScript frontend (Next.js + Vercel AI SDK)
                                       │
                                       ▼
                  ┌─────── EDGE / GATEWAY ─────────┐
                  │  CloudFront + ALB                │  ← M46/M47 (k8s + ingress)
                  │  OIDC verify (Auth0)             │  ← M80 §8
                  │  Rate-limit (LiteLLM)            │  ← M80 §6, M79
                  │  Tenant-tag injection             │  ← M80 §3
                  └────────────────┬────────────────┘
                                   │
                  ┌────────────────▼─────────────────┐
                  │           CONTROL PLANE           │
                  │  FastAPI gateway (M28) + Go svcs   │  ← M52
                  │  Workflows on LangGraph (M33)      │
                  │  MCP tools (M64)  ·  A2A peers     │  ← M75
                  │  Guard model (M74)                  │
                  │  Cost meter (M79)                   │
                  └─┬──────────┬───────────┬───────────┘
                    │          │           │
        ┌───────────▼──┐    ┌──▼─────────┐ │     ┌────────────────────┐
        │ Retrieval    │    │ Inference  │ │     │  Async workers      │
        │ Qdrant (M42) │    │ vLLM       │◄┼─────│  Celery / Redis Q   │
        │ Postgres +   │    │ + multi-   │ │     │  embedding · OCR    │
        │ pgvector(M36)│    │ LoRA (M80) │ │     │  fine-tune triggers │
        │ Feast (M69)  │    │ + KV cache │ │     └──────────┬──────────┘
        │ Redis (M37)  │    │ (M60)      │ │                │
        │              │    │ + cache    │ │                ▼
        │ Reranker     │    │ tier — vLLM/│ │     ┌────────────────────┐
        │ (cross-enc)  │    │ Triton/OAI │ │     │  Spark + Delta (M73)│
        │              │    │ /Anthro    │ │     │  Bronze · Silver ·  │
        └──────────────┘    └──┬─────────┘ │     │  Gold features      │
                               │            │     └──────────┬──────────┘
                               ▼            │                │
                   ┌──────────────────────┐ │                ▼
                   │ Hosted LLMs           │ │     ┌────────────────────┐
                   │ Claude · GPT · Gemini │ │     │  Postgres (auth,   │
                   │ (tier-routed M79)     │ │     │  audit, billing)   │
                   └──────────────────────┘ │     └────────────────────┘
                                            │
                                            ▼
                          ┌──────────────────────────────────────┐
                          │  Observability stack                  │
                          │  Prometheus (M50) · Grafana           │
                          │  Loki · Tempo · OTel Collector (M51)  │
                          │  Langfuse — LLM traces + cost (M70)   │
                          │  Sentry — error tracking              │
                          └──────────────────────────────────────┘
                                            │
                                            ▼
                          ┌──────────────────────────────────────┐
                          │  Platform plumbing                    │
                          │  Terraform (M48) · Helm (M47)         │
                          │  Argo CD · GitHub Actions             │
                          │  Vault · AWS Secrets Manager          │
                          │  Datadog APM (regulated tenants)      │
                          └──────────────────────────────────────┘
```

**One picture, ~25 components, every box maps to one or two modules.** Print it out, pin it next to your desk.

## 3 · Layer-by-layer module mapping

| Layer | Choice | Primary modules | Why |
|---|---|---|---|
| **Frontend** | Next.js + Vercel AI SDK | **M54** | streaming chat, tool-call UI in <1 day |
| **API gateway** | FastAPI + LiteLLM | **M28**, **M79**, **M80** §3 | one place to auth, rate-limit, meter, route |
| **Auth / IdP** | Auth0 (later: Cognito for AWS native) | **M80** §8 | OIDC + per-tenant scopes |
| **Orchestration** | LangGraph + Pydantic AI | **M33**, **M35**, **M59** | stateful agents, structured outputs |
| **Tools** | MCP servers (filesystem, ServiceNow, Slack, GitHub) | **M64** | one tool standard across customers |
| **Inter-agent** | A2A for partner integrations | **M75** | optional Q3+ for enterprise tier |
| **Inference** | vLLM with `--enable-lora`, fallback Triton for vision | **M44**, **M60**, **M71**, **M80** §5 | PagedAttention + per-tenant LoRA on one base |
| **Hosted LLMs** | Anthropic Claude Sonnet 4.6 (primary), GPT-5 mini (router), Gemini Flash (long-ctx) | **M79** §4 | best for code/agents, cheap router, big context |
| **Quantization** | INT4-AWQ for self-hosted, INT8 KV | **M38** §2 | 3× throughput at <2% quality loss |
| **Retrieval** | Qdrant (vectors) + pgvector (small tenants) + BM25 (Postgres `tsvector`) | **M42**, **M36** | hybrid; filter-per-tenant |
| **Reranker** | Cohere reranker (API) | **M30** | quality > self-host for v1 |
| **Feature store** | Feast on Postgres + Redis | **M69** | user features, RAG metadata |
| **Cache** | Redis (response + prefix + LoRA hot-set) | **M37** | response cache = top cost lever |
| **Data lake** | S3 + Delta Lake | **M73** | Bronze (raw ingests) / Silver (cleaned) / Gold (features) |
| **Streaming** | Kinesis → Spark Structured Streaming | **M73** §9 | doc-change events into the index |
| **Batch + ETL** | Dagster | **M68** | asset-based; lineage matters for HIPAA |
| **Fine-tuning** | Unsloth + QLoRA on g5.12xlarge | **M39**, **M58**, **M59** | per-customer LoRA; cheap |
| **Evals** | Promptfoo in CI + Langfuse online evals | **M70** | gate every release; track per-tenant quality |
| **Synthetic data** | Stronger model → SFT dataset for cheap model | **M88** (planned) | reduce hosted LLM bill 5-10× |
| **Security** | OWASP LLM Top 10 + garak in CI + Llama Guard 3 | **M74** | non-negotiable for hospitals/banks |
| **Multi-tenancy** | namespaced k8s + per-tenant LoRA, vector collection per tenant | **M80** | day-1 design |
| **k8s** | EKS (AWS) + AKS (Azure regulated tenant) | **M46** | one Helm chart for both |
| **IaC** | Terraform (infra) + Helm (apps) + Ansible (AMIs) | **M47**, **M48**, **M49** | every prod change reviewable |
| **Observability** | Prometheus + Grafana + Loki + Tempo + OTel + Langfuse + Sentry | **M50**, **M51**, **M70** | tenant_id labels everywhere |
| **FinOps** | Langfuse + OpenCost + LiteLLM budgets + AWS Cost Anomaly | **M79** | $/user/day dashboard from day 1 |
| **Backbone services** | Go for sidecars + gateways; Rust for one hot vector-search proxy | **M52**, **M53** | static binary cold start; perf where it counts |
| **Compliance** | SOC2 via Vanta, HIPAA via AWS BAA, audit trail in S3 Object Lock | **M74**, **M80** §9 | non-negotiable |
| **AI ops** | Slack alerts, on-call rotation, runbooks | **M50** §10 | day-1 |

> 🟡 **What we deliberately defer past v1.** Distributed pretraining (M66), bare-metal (M78), GPU networking (M77), MoE / DPO from-scratch (M62 internals). We use these *components* (HF safetensors, LoRA, vLLM) but don't run our own pretrain. If we ever do, M55-M67 are the playbook.

## 4 · The repo layout

```
acme-ai/
├── apps/
│   ├── web/                     # Next.js frontend (M54)
│   ├── gateway/                 # FastAPI + LiteLLM (M28, M79)
│   ├── workers/                 # Celery / async (M28)
│   └── mcp-servers/             # Internal MCP servers (M64)
│       ├── filesystem/
│       ├── servicenow/
│       └── slack/
├── packages/
│   ├── agents/                  # LangGraph workflows (M33, M35)
│   ├── retrieval/               # Qdrant + pgvector + BM25 (M42, M36)
│   ├── eval/                    # Promptfoo + Langfuse harness (M70)
│   ├── guardrails/              # Llama Guard 3 + PII redaction (M74)
│   └── shared/                  # cost meter, tenant ctx, OTel (M51, M79, M80)
├── ml/
│   ├── notebooks/               # research / data analysis
│   ├── finetune/                # QLoRA / Unsloth jobs (M39)
│   ├── synthetic/               # data generation (M88)
│   └── datasets/                # version-pinned eval sets
├── infra/
│   ├── terraform/               # AWS + Azure (M48)
│   │   ├── modules/
│   │   ├── envs/{dev,staging,prod}
│   │   └── ...
│   ├── helm/                    # one chart, value overrides per env (M47)
│   ├── argocd/                  # apps-of-apps (M47)
│   └── ansible/                 # base AMI baking (M49)
├── data/
│   ├── dbt/                     # transformations
│   ├── dagster/                 # pipelines (M68)
│   └── delta-tables/            # Bronze / Silver / Gold (M73)
├── observability/
│   ├── prometheus/              # rules, recording, alerts (M50)
│   ├── grafana/                 # dashboards as JSON (M50)
│   └── otel-collector/          # config (M51)
├── security/
│   ├── policies/                # OPA / Kyverno
│   ├── redteam/                 # garak / Promptfoo red-team (M74)
│   └── runbooks/                # incident response
└── docs/
    ├── adr/                     # Architecture Decision Records
    ├── runbooks/
    └── per-tenant/
```

**Why a monorepo for v1.** One PR can touch UI + gateway + agent + infra. Less ceremony, faster iteration. Split when the team passes ~25 engineers (you won't in Year 1).

## 5 · A live request walkthrough — 17 hops, 1.4 s budget

One real chat turn. Aim for **p99 < 2.5 s**; budget per layer below adds to ~1.4 s steady-state.

```
   User types "What's the deductible for customer #4471?" → presses Enter

   1   browser              streaming POST /api/chat                     (5 ms)
   2   Vercel edge          add request id, forward                       (10 ms)
   3   FastAPI gateway      verify Auth0 JWT, extract tenant_id           (8 ms)
   4   LiteLLM router       per-tenant rate-limit check (Redis)           (3 ms)
   5   Guard model          Llama Guard 3 — inject scan                   (40 ms)
   6   LangGraph workflow   route → "search-then-answer"                   (1 ms)
   7   Retriever            BM25 + Qdrant hybrid; tenant filter           (35 ms)
   8                         re-rank via Cohere API                       (60 ms)
   9   Tool decision        none needed                                    (1 ms)
   10  Prompt builder       system + retrieved chunks + few-shot          (5 ms)
   11  Prompt cache check   Redis hit ratio ~0.5 prefix cached            (8 ms)
   12  vLLM dispatch        Claude Sonnet 4.6 (router picked this tier)    (—)
   13  Anthropic API        streaming response                           (~900 ms first byte)
   14  Output filter        PII redact + Llama Guard 3 — output           (15 ms)
   15  Tool calls (if any)  ServiceNow create-ticket — async via worker   (—)
   16  Cost meter           track tokens × $/Mtok in Langfuse + OTel     (3 ms)
   17  Response             stream tokens to client                       (cont.)

   Total first byte to user:  ~1.1 s
   p99 budget:                2.5 s
```

The biggest line item is the LLM API call. Two levers to pull as scale grows:
- **prompt cache** (Anthropic offers 90 % off cached input) — top-line saving once shared system prompts stabilise.
- **router escalation** — answer 60 % of routes with Haiku / GPT mini at a tenth the cost. The "policy_simple_classify" model is itself a small fine-tune.

**The capstone integration test:** assert each hop with **OTel trace spans + tenant_id attribute**. M51 made this trivial — now it's the single-pane debugging surface.

## 6 · The 26-week build plan

```
Weeks 0-4   FOUNDATION                                              · M46 M47 M48
            ┌─ EKS dev cluster, Argo CD, Terraform modules
            ├─ Auth0 tenant model, FastAPI gateway skeleton           · M28
            ├─ Postgres + Redis + Qdrant
            └─ OTel collector + Prometheus + Grafana                  · M50 M51

Weeks 4-8   v0  CORE CHAT (single tenant, hosted-LLM only)           · M30 M32
            ┌─ Naive RAG (Qdrant, no rerank)
            ├─ Anthropic Claude Sonnet via LiteLLM                    · M79
            ├─ Streaming response, Next.js UI                         · M54
            └─ Promptfoo CI smoke evals                               · M70

Weeks 8-14  v1  PRODUCTION HARDENING                                  · M74 M50 M80
            ┌─ Multi-tenant (namespaced + per-customer Qdrant tenant) · M80
            ├─ OIDC scopes, per-tenant API keys                       · M80 §8
            ├─ Llama Guard 3 input + output filters                   · M74
            ├─ Cost meter (Langfuse), $/user/day Grafana panel        · M79
            ├─ Hybrid search (BM25 + vector), rerank API              · M30 M42
            ├─ MCP tools — filesystem + ServiceNow + Slack            · M64
            ├─ HITL approval flow for write actions                   · M33
            └─ Red-team CI gate (garak + Promptfoo)                   · M74

Weeks 14-20 v2  PER-TENANT LoRA + INFERENCE COST                     · M39 M44 M60
            ┌─ Unsloth QLoRA pipeline on g5.12xlarge                  · M39
            ├─ vLLM cluster (self-host Llama 3.3 70B) with --enable-lora· M44 M80
            ├─ Cheap-first router (Haiku for simple → Sonnet)         · M79
            ├─ Prompt + response cache in Redis                       · M37
            ├─ Per-tenant LoRA hot-swap                               · M80 §5
            └─ Anthropic + OpenAI as failover                         · M79

Weeks 20-24 v3  ENTERPRISE READINESS                                  · M68 M73 M75
            ┌─ Dagster pipelines for tenant ingestion                 · M68
            ├─ Spark + Delta Lake for analytics + audit               · M73
            ├─ Azure AKS deploy for one regulated tenant              · M46
            ├─ HIPAA: AWS BAA + S3 Object Lock audit logs             · M80
            ├─ SOC2 evidence via Vanta + Drata
            ├─ Tenant lifecycle automation (Terraform module per tenant)· M48 M80
            ├─ A2A endpoint for one partner integration               · M75
            └─ Failure-injection drills (Xid, AZ outage)              · M78

Weeks 24-26 GA
            ┌─ Customer onboarding playbook
            ├─ On-call rotation + runbooks
            ├─ Capacity planning (next 90 days)
            └─ Pricing + contracts finalised
```

Twelve milestones; six months end-to-end; every one references prior modules.

## 7 · The team

Eight engineers is the practical minimum. One lead + seven specialists:

| Role | Headcount | Owns | Lean-on modules |
|---|---|---|---|
| **Platform Lead** | 1 | architecture, on-call rotation, hiring | all of them |
| **Backend / API** | 1 | FastAPI gateway, MCP servers, auth | M28, M32, M64, M80 |
| **ML / Inference** | 1 | vLLM, fine-tuning, evals, cost | M39, M44, M60, M70, M79 |
| **Data / Retrieval** | 1 | Qdrant, BM25, Feast, Dagster, Spark | M30, M42, M68, M69, M73 |
| **Frontend** | 1 | Next.js, streaming chat, settings UX | M54 |
| **Infra / SRE** | 1 | Terraform, Helm, Argo CD, K8s | M46, M47, M48 |
| **Observability + Security** | 1 | Prometheus, OTel, Langfuse, guardrails | M50, M51, M70, M74 |
| **Customer Success engineer** | 1 | tenant onboarding, ingestion, white-glove | M80 lifecycle |

In Year 2 each of those becomes a team of 2-4.

## 8 · Day-1 launch checklist

Before turning the GA switch on, **every one** of these must be green:

### Security (M74)
- [ ] OWASP LLM Top 10 review signed off
- [ ] Llama Guard 3 input + output filters in every chat route
- [ ] Garak + Promptfoo red-team in CI; block deploy on regression
- [ ] All secrets in Vault / Secrets Manager (no env vars in repos)
- [ ] mTLS between internal services
- [ ] Pen test done by external firm

### Multi-tenancy (M80)
- [ ] Tenant filter test on every retriever path
- [ ] Per-tenant LoRA signed at training, verified at load
- [ ] HIPAA / SOC2 audit-log pipeline writing to S3 Object Lock
- [ ] DSAR / right-to-erasure runbook tested end-to-end
- [ ] Per-tenant Grafana dashboard auto-provisioned

### Observability (M50, M51, M70)
- [ ] OTel `tenant.id` attribute on every span
- [ ] Cost-per-tenant + $/user/day Grafana panels
- [ ] Langfuse session tracking + online eval scorers
- [ ] Alerts: p99 latency, error rate, cost spike, GPU OOM, vector-DB latency
- [ ] On-call rotation + 24/7 PagerDuty for SEV-1/SEV-2

### Reliability
- [ ] Game-day chaos test (kill a vLLM pod mid-stream)
- [ ] AZ-failover drill (us-east-1a → 1b)
- [ ] Failover plan from Anthropic to OpenAI to Bedrock
- [ ] Backup / restore tested for every stateful store
- [ ] Disaster Recovery runbook (RPO < 1h, RTO < 4h)

### FinOps (M79)
- [ ] Per-tenant budget alert in Slack (1.5× moving avg)
- [ ] Per-route $/Mtok dashboard
- [ ] Spot capacity for batch workloads
- [ ] Cache hit-rate target = 50%+ in steady-state

### Product
- [ ] First customer onboarding playbook tested by Customer Success
- [ ] Pricing approved by Finance
- [ ] Contract templates approved by Legal
- [ ] Support docs published

Twenty-eight items. None optional.

## 9 · Day-90 evolution — what to add next

Post-GA the system runs; now you scale. **What to add in Year 2:**

| Area | Module(s) | Why |
|---|---|---|
| **Self-host top route** | M44, M60, M71, M79 | hosted bill is now ≥ $200k/mo; vLLM cluster pays for itself |
| **DPO on top routes** | M62 | preference fine-tunes from production thumbs-up/down |
| **GraphRAG** | M87 | citations + multi-hop reasoning |
| **Synthetic-data factory** | M88 | distil strong model → cheap model continuously |
| **Constitutional / RLAIF** | M89 | safer for hospitals |
| **Edge / on-device summarizer** | M90 | offline mobile for field reps |
| **Computer-use agent** | M72 | "open this customer's CRM, find Lisa's account, schedule a follow-up" |
| **Multimodal** | M65 | ingest screenshots + voice memos |
| **Bare-metal eval** | M78 | run the bare-metal-vs-cloud crossover for your actual usage |
| **GPU networking** | M77 | when you sign Year-2 capacity contracts with cloud or co-lo |

Don't pre-build. Earn each addition with a clear product or cost-of-the-bill rationale.

## 10 · Failure modes that will hit you

Pre-loaded so you don't have to learn them by suffering:

| Failure | Cause | Module / Fix |
|---|---|---|
| **A tenant's RAG returns another tenant's docs** | missing filter on one path | M80 — integration test on every retriever call |
| **One tenant's traffic eats the whole cluster** | no fair-share scheduling | M80 — Volcano / Kueue; Redis sliding window |
| **Vendor pricing changes overnight** | Anthropic 2× output cost in Q3 | M79 — abstract via LiteLLM; fall back to GPT-5 mini |
| **Llama Guard misclassifies a clinical question** | shallow safety training on medical text | M74 — fine-tune a guard model on your domain |
| **Cost runs 3× projection** | streaming responses uncapped | M79 — `max_tokens` ceiling + per-tenant budget |
| **Latency p99 = 8s** | retriever serial calls + no rerank cache | M30 — parallelise + cache rerank results |
| **Xid 79 mid-training** | bad GPU in a fine-tune fleet | M78 — auto-drain + checkpoint resume |
| **HIPAA audit fails** | logs leak PHI to Sentry | M74 — PII redaction in OTel processor |
| **Prompt cache poisoning** | shared key collision | M74 — namespace cache keys by tenant_id |
| **Customer says "delete me"** | GDPR DSAR | M80 — Terraform-driven tenant teardown tested in dev quarterly |

Most production incidents reduce to one of these ten. Have a runbook for each.

## ✅ Recap

- The **reference architecture** is one A4 picture: edge → gateway → orchestration → tools → retrieval → inference → data lake → observability → platform plumbing.
- Every layer maps to **1-3 prior modules**. M1-M80 was the parts catalogue; this module is the wiring diagram.
- The **26-week plan** ships v0 → v3 → GA in six months with a team of 8.
- The **day-1 checklist** is 28 boxes, none optional, all backed by earlier modules.
- The **failure-mode catalogue** is 10 incidents that every platform meets — runbooks ready.

🎓 **You've reached the end of Phase 10 Capstone.** What follows in Parts 11-12 (M82-M90) is depth and breadth on top — Linux + Bash, CNN/RNN/GAN deep dives, GraphRAG, synthetic data, Constitutional AI, Edge AI. Each makes the platform you just built smarter, cheaper, or safer.
